# M5 — trânsito físico Bayesiano

Interface fina para a implementação canônica em `src/bayesian_modeling/physical_transit.py`. O notebook não contém uma segunda implementação do modelo e não modifica RAW, Silver ou Gold.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from bayesian_modeling.contracts import build_m5_paths
from bayesian_modeling.physical_transit import run_m5
from project_config import get_target


In [ ]:
target_slug = 'kepler_10_b'
run_id = 'scientific_002'
target = get_target(target_slug)
paths = build_m5_paths(PROJECT_ROOT, target, run_id)
target.metadata()


## Execução opcional

O full MCMC não roda implicitamente. Ative a flag somente no ambiente científico fixado.


In [ ]:
RUN_FULL_SAMPLING = False
if RUN_FULL_SAMPLING:
    run_m5(
        target_slug=target_slug,
        run_id=run_id,
        prior_profile_name='baseline',
        sampling_overrides={
            'draws': 800, 'tune': 800, 'chains': 4,
            'cores': 4, 'target_accept': 0.95,
        },
    )


## Inspeção dos artefatos e do gate


In [ ]:
status = json.loads((paths.model_dir / 'run_status.json').read_text(encoding='utf-8'))
config = json.loads(paths.model_config_path.read_text(encoding='utf-8'))
posterior = pd.read_csv(paths.posterior_summary_path)
derived = pd.read_csv(paths.derived_summary_path)
{
    'status': status['status'],
    'target': config['target']['planet_name'],
    'dataset_id': config['input_summary']['dataset_id'],
    'diagnostics': config['diagnostics'],
    'interpretation_gate': config['interpretation_gate'],
}


In [ ]:
posterior, derived
